# Declaration of Originality

**School of Informatics & IT**
<br/>**Diploma in Applied Artificial Intelligence**
<br/>**Machine Learning for Developers (CAI2C08)**
<br/>**AY2026/2027 April Semester**
<br/>**Program Codes**

* Student Name: Eric Ng Eng Chee



**Declaration of Originality**
* I am the originator of this work, and I have appropriately acknowledged all other original sources used as my references for this work.
* I understand that Plagiarism is the act of taking and using the whole or any part of another person’s work, including work generated by AI, and presenting it as my own.
* I understand that Plagiarism is an academic offence and if I am found to have committed or abetted the offence of plagiarism in relation to this submitted work, disciplinary action will be enforced.

# Libraries

In [1]:
## Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

# 1. Business Understanding



**Problem**
First time HDB buyers do not know what a fair price for HDB flats in Singapore are. This is because HDB prices can change a lot depending many factors like town, flat type, floor area, which storey it is and how much lease is left. This can cause people to overpay for their HDB by thousands of dollars.

**Target Audience**
First time HDB buyers across Singapore who have shortlisted a flat and want a quick, data based estimate before making an offer.

**Solution**
The solution is a machine learning model that predicts a fair resale price for any HDB flat in Singapore, given its details (town, flat type, floor area, storey, lease years left, etc.). The model is then implemented onto Streamlit as a web app where user picks the flat's details from dropdowns and sliders, and the app returns an estimated fair price they can compare against the seller's asking price.

**Regression**
The target "resale_price" is a continuous dollar amount, not a category, so this is a regression problem. The output has to be usable directly as a dollar value.

**Dataset**
Public HDB resale transaction data from "data.gov.sg". Covers every completed resale from January 2017 onwards (~234,000 rows) across all 26 HDB towns in Singapore. Real, non synthetic data.

**Why it matters**
Right now, most first time HDB buyers have nothing to compare an asking price against, so it is hard to tell a fair deal from an overpriced one. The solution, helps to give a quick, data based estimate to give them a benchmark. If the seller is asking far above the estimate, that is a clear signal to negotiate or walk away. Even a rough estimate is useful, because it turns a blind guess into an informed decision on a purchase worth hundreds of thousands of dollars.

# 2. Data Understanding

## 2.1 Load dataset

In [2]:
## Read *.csv file into pandas DataFrame
FILE_PATH = "ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv"
df = pd.read_csv(FILE_PATH)
df

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0
...,...,...,...,...,...,...,...,...,...,...,...
234667,2026-03,YISHUN,EXECUTIVE,836,YISHUN ST 81,10 TO 12,146.0,Maisonette,1988,61 years,995000.0
234668,2026-03,YISHUN,EXECUTIVE,877,YISHUN ST 81,07 TO 09,142.0,Apartment,1987,60 years 10 months,980000.0
234669,2026-04,YISHUN,EXECUTIVE,827,YISHUN ST 81,01 TO 03,145.0,Maisonette,1987,60 years 06 months,960000.0
234670,2026-05,YISHUN,EXECUTIVE,828,YISHUN ST 81,07 TO 09,145.0,Apartment,1988,60 years 09 months,1068888.0


## 2.2 Summary Statistics

In [3]:
## Understand the type of variable for each column
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234672 entries, 0 to 234671
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                234672 non-null  object 
 1   town                 234672 non-null  object 
 2   flat_type            234672 non-null  object 
 3   block                234672 non-null  object 
 4   street_name          234672 non-null  object 
 5   storey_range         234672 non-null  object 
 6   floor_area_sqm       234672 non-null  float64
 7   flat_model           234672 non-null  object 
 8   lease_commence_date  234672 non-null  int64  
 9   remaining_lease      234672 non-null  object 
 10  resale_price         234672 non-null  float64
dtypes: float64(2), int64(1), object(8)
memory usage: 19.7+ MB


In [4]:
## Check for missing data
df.isnull().sum()

month                  0
town                   0
flat_type              0
block                  0
street_name            0
storey_range           0
floor_area_sqm         0
flat_model             0
lease_commence_date    0
remaining_lease        0
resale_price           0
dtype: int64

In [5]:
## Describe data distribution
df.describe(include="all")

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
count,234672,234672,234672,234672,234672,234672,234672.000000,234672,234672.000000,234672,2.346720e+05
unique,115,26,7,2768,578,17,NaN,21,NaN,700,NaN
top,2024-07,SENGKANG,4 ROOM,2,YISHUN RING RD,04 TO 06,NaN,Model A,NaN,94 years 10 months,NaN
freq,3036,19037,99637,707,3318,53801,NaN,84379,NaN,1924,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,96.688870,NaN,1996.559867,NaN,5.317508e+05
std,NaN,NaN,NaN,NaN,NaN,NaN,24.017378,NaN,14.364676,NaN,1.906333e+05
min,NaN,NaN,NaN,NaN,NaN,NaN,31.000000,NaN,1966.000000,NaN,1.400000e+05
25%,NaN,NaN,NaN,NaN,NaN,NaN,81.000000,NaN,1985.000000,NaN,3.900000e+05
50%,NaN,NaN,NaN,NaN,NaN,NaN,93.000000,NaN,1997.000000,NaN,5.000000e+05
75%,NaN,NaN,NaN,NaN,NaN,NaN,112.000000,NaN,2012.000000,NaN,6.388880e+05


In [6]:
## Count unique values per column
df.nunique()

month                   115
town                     26
flat_type                 7
block                  2768
street_name             578
storey_range             17
floor_area_sqm          192
flat_model               21
lease_commence_date      57
remaining_lease         700
resale_price           4697
dtype: int64

## 2.3 Data Visualization

### 2.3.1 Understanding distribution of data

### 2.3.1.1 Understanding distribution of target

In [ ]:
## Understanding distribution of target

# Histogram
df['resale_price'].hist(bins=50)
plt.xlabel('Resale Price (SGD)')
plt.ylabel('Frequency')
plt.title('Distribution of HDB Resale Prices')
plt.show()
## interpretation:
# The histogram shows that the resale price distribution is right skewed, with most
# flats selling between ~$390k and ~$640k (the middle 50%). A long tail stretches past
# $1m, showing a small number of premium transactions well above the typical price.

# Boxplot
df['resale_price'].plot(kind='box')
plt.ylabel('Resale Price (SGD)')
plt.title('Boxplot of HDB Resale Prices')
plt.show()
## interpretation:
# The boxplot confirms the right skew, with a median resale price of ~$500k and many
# outliers sitting above the upper whisker (up to ~$1.73m). These outliers are real
# premium transactions (large flats, high storey / DBSS units), not data errors, so
# they should be kept in the modelling data.

### 2.3.1.2 Understanding distribution of features

In [ ]:
## Understanding distribution of features

# Numeric features (histograms)
numeric_cols = ['floor_area_sqm', 'lease_commence_date']
df[numeric_cols].hist(bins=30, figsize=(12, 4))
plt.show()
## interpretation:
# The histograms show that floor_area_sqm is roughly bell shaped and centered around
# 90 to 100 sqm (driven by the many 4 room and 5 room flats). lease_commence_date spans
# 1966 to 2022 fairly evenly, so the dataset covers both old mature estate flats and
# newer BTOs.

# Numeric features (boxplots)
df[numeric_cols].plot(kind='box', subplots=True, layout=(1, 2), figsize=(12, 4))
plt.show()
## interpretation:
# The boxplots show that floor_area_sqm has outliers on the high end (executive flats
# up to ~370 sqm) and a small number on the low end (1/2 room flats). lease_commence_date
# is fairly symmetric with no strong outliers. The high area outliers are genuine premium
# units, not data errors, so they should be kept in the modelling data.

# Categorical features (horizontal bar charts so long labels don't overlap)
categorical_cols = ['town', 'flat_type', 'flat_model', 'storey_range']
for col in categorical_cols:
    df[col].value_counts().sort_values().plot(kind='barh', figsize=(8, 6))
    plt.title(f'Distribution of {col}')
    plt.xlabel('Count')
    plt.show()
## interpretation:
# The bar charts show that sales are heavily concentrated in a few categories. SENGKANG,
# PUNGGOL, WOODLANDS, TAMPINES and YISHUN dominate the towns, while BUKIT TIMAH has only
# ~570 rows. 4 ROOM is by far the most common flat_type (~40% of rows), with 1 ROOM and
# MULTI GENERATION having <100 rows each. 'Model A' and 'Improved' dominate flat_model,
# and most transactions are on floors 01 to 15 with very few above floor 30. The model will
# predict best for these common categories and less reliably for rare ones.

### 2.3.2 Understanding relationship between variables

In [ ]:
## Understanding relationship between variables

# Scatter: floor_area_sqm vs resale_price
plt.figure(figsize=(8, 5))
plt.scatter(df['floor_area_sqm'], df['resale_price'], alpha=0.1)
plt.xlabel('Floor Area (sqm)')
plt.ylabel('Resale Price (SGD)')
plt.title('Floor Area vs Resale Price')
plt.show()
## interpretation:
# The scatter plot shows a strong positive, roughly linear relationship between
# floor_area_sqm and resale_price. Larger flats sell for more, so even a simple linear
# model should be able to capture most of this signal.

# Scatter: lease_commence_date vs resale_price
plt.figure(figsize=(8, 5))
plt.scatter(df['lease_commence_date'], df['resale_price'], alpha=0.1)
plt.xlabel('Lease Commence Year')
plt.ylabel('Resale Price (SGD)')
plt.title('Lease Commence Year vs Resale Price')
plt.show()
## interpretation:
# The scatter plot shows a positive but weaker relationship between lease_commence_date
# and resale_price. There is a large vertical spread at every year, meaning other factors
# (town, flat type, storey) also strongly influence price.

# Boxplot: town vs resale_price (horizontal so all 26 town labels stay readable)
plt.figure(figsize=(10, 8))
sns.boxplot(x='resale_price', y='town', data=df)
plt.title('Resale Price by Town')
plt.show()
## interpretation:
# The boxplot shows that town is a major driver of price. Central and mature towns
# (BUKIT TIMAH ~$777k, BISHAN ~$695k, QUEENSTOWN ~$670k, BUKIT MERAH ~$660k) have the
# highest medians, while outer towns (ANG MO KIO ~$418k, YISHUN ~$432k, BEDOK ~$433k)
# have the lowest. That is nearly a $360k median gap between the extremes, so town must
# stay as an input feature in the model.

# Boxplot: flat_type vs resale_price
plt.figure(figsize=(10, 5))
sns.boxplot(x='flat_type', y='resale_price', data=df)
plt.title('Resale Price by Flat Type')
plt.show()
## interpretation:
# The boxplot shows a clear step up in resale price from 1 ROOM through EXECUTIVE.
# Larger flat types cost more, as expected. MULTI GENERATION flats also sit at the high
# end (they are essentially oversized executives). flat_type is clearly informative on
# its own.

# Boxplot: storey_range vs resale_price (horizontal, ordered from lowest floor to highest)
storey_order = sorted(df['storey_range'].unique())
plt.figure(figsize=(10, 7))
sns.boxplot(x='resale_price', y='storey_range', data=df, order=storey_order)
plt.title('Resale Price by Storey Range')
plt.show()
## interpretation:
# The boxplot shows a clear, near monotonic rise in resale price as storey increases.
# The median goes from ~$450k on floors 01 to 03 up to ~$1.23m on floors 49 to 51. Storey is a
# strong ordinal feature, so its bucketed string form ("10 TO 12") should be turned into
# a numeric feature (the bucket midpoint) so the model can use the ordering.

# Boxplot: flat_model vs resale_price (horizontal, ordered by median price descending)
model_order = df.groupby('flat_model')['resale_price'].median().sort_values(ascending=False).index.tolist()
plt.figure(figsize=(10, 7))
sns.boxplot(x='resale_price', y='flat_model', data=df, order=model_order)
plt.title('Resale Price by Flat Model')
plt.show()
## interpretation:
# The boxplot shows a very wide price range across flat models. Premium models (Type S2
# ~$1.13m, Type S1 ~$1.02m, Premium Apartment Loft ~$965k) command large premiums, while
# common models like Standard (~$345k), 2 room (~$365k) and New Generation (~$380k) sit
# at the low end. flat_model is a strong categorical feature and should be one hot encoded
# so the model can price these levels separately.

# Line: median resale price by year (using the first 4 characters of month = YYYY)
df.groupby(df['month'].str[:4])['resale_price'].median().plot(figsize=(10, 4), marker='o')
plt.title('Median HDB Resale Price by Year')
plt.ylabel('Median Resale Price (SGD)')
plt.xlabel('Year')
plt.show()
## interpretation:
# The line chart shows the Singapore HDB market has trended sharply upwards over
# 2017 to 2026. The median resale price rose from ~$410k in 2017 to ~$630k in 2026, a ~54%
# increase, with a clear acceleration from 2020 onwards. This confirms that a
# transaction year feature is needed so the model can tell an older sale from a recent
# one at the same flat.

# Correlation heatmap of numeric variables
plt.figure(figsize=(6, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()
## interpretation:
# The heatmap shows that floor_area_sqm is the strongest linear correlate with
# resale_price (r ~ 0.57), followed by lease_commence_date (r ~ 0.37). Pearson r only
# measures linear association, so tree based models later may still find useful
# non linear structure that a linear model misses.

# 3. Data Preparation

Before we can train any model, the data needs to be in a shape the models can actually use. Right now it has some duplicate rows, columns that are just IDs, and text columns that models cannot do maths on. We fix all of that here, in four steps:

1. **Data cleaning.** Remove duplicate rows and drop ID style columns (block and street name) that do not help predict price.
2. **Feature engineering.** Turn text columns that really hold numbers (remaining lease, storey range, month) into proper number columns.
3. **One hot encoding.** Turn the leftover word columns (town, flat type, flat model) into 0/1 columns, because models only understand numbers, not words.
4. **Train test split.** Split the data into a training set (to teach the model) and a test set (kept hidden, to fairly check how well the model does on flats it has never seen).

## 3.1 Data Cleaning

Before modelling, we make sure every row is clean and useful:

1. **Missing values** are checked again. Section 2.2 already showed there are none, so no imputation or row removal is needed.
2. **Duplicate rows** are checked. Any rows that are identical across every column are dropped so that each row counts as one unique transaction.
3. **ID like columns** (`block` and `street_name`) are dropped. They have very high cardinality and act mainly as location identifiers, and `town` already captures location well enough for the model.

Note that we deliberately keep the high price outliers seen during EDA, since those are real premium transactions. Removing them would make the model underprice large or high floor flats.

In [10]:
## Clean data

# Check for missing values (we saw 0 earlier, just confirming).
print("Total missing values:", df.isnull().sum().sum())

# Find and remove exact duplicate rows so each row is one unique sale.
print("Duplicate rows found:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows after cleaning:", df.duplicated().sum())

# Drop block and street_name. They have too many different values to be useful,
# and town already tells us the location.
df = df.drop(['block', 'street_name'], axis=1)

print("Shape after cleaning:", df.shape)
df.head()

Total missing values: 0
Duplicate rows found: 316
Duplicate rows after cleaning: 0
Shape after cleaning: (234356, 9)


,month,town,flat_type,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [11]:
## Feature engineering
# Turn the text columns that really hold numbers into actual numbers.

# "61 years 04 months" -> 61.33 years
def lease_to_years(text):
    parts = text.split()
    years = int(parts[0])
    months = int(parts[2]) if len(parts) > 2 else 0   # some rows have no months
    return years + months / 12

df['remaining_lease_years'] = df['remaining_lease'].apply(lease_to_years)

# "10 TO 12" -> 11 (the middle floor)
def storey_to_mid(text):
    low, high = text.split(' TO ')
    return (int(low) + int(high)) // 2

df['storey_mid'] = df['storey_range'].apply(storey_to_mid)

# take the first 4 characters of the month (the year the flat was sold)
df['txn_year'] = df['month'].str[:4].astype(int)

# Drop the old text columns now that we have number versions.
# Also drop lease_commence_date, because remaining_lease_years and txn_year
# already cover the same info (keeping all three confuses the linear model).
df = df.drop(['month', 'remaining_lease', 'storey_range', 'lease_commence_date'], axis=1)

print("Columns after feature engineering:", df.columns.tolist())
df.head()

Columns after feature engineering: ['town', 'flat_type', 'floor_area_sqm', 'flat_model', 'resale_price', 'remaining_lease_years', 'storey_mid', 'txn_year']


,town,flat_type,floor_area_sqm,flat_model,resale_price,remaining_lease_years,storey_mid,txn_year
0,ANG MO KIO,2 ROOM,44.0,Improved,232000.0,61.333333,11,2017
1,ANG MO KIO,3 ROOM,67.0,New Generation,250000.0,60.583333,2,2017
2,ANG MO KIO,3 ROOM,67.0,New Generation,262000.0,62.416667,2,2017
3,ANG MO KIO,3 ROOM,68.0,New Generation,265000.0,62.083333,5,2017
4,ANG MO KIO,3 ROOM,67.0,New Generation,265000.0,62.416667,2,2017


In [12]:
## One hot encoding
# Models cannot read words, so we turn the text columns (town, flat_type,
# flat_model) into 0/1 columns. drop_first=True drops one column from each
# group to avoid repeating the same information.
df = pd.get_dummies(df, columns=['town', 'flat_type', 'flat_model'], drop_first=True)

print("Shape after encoding:", df.shape)
df.head()

Shape after encoding: (234356, 56)


,floor_area_sqm,resale_price,remaining_lease_years,storey_mid,txn_year,town_BEDOK,town_BISHAN,town_BUKIT BATOK,town_BUKIT MERAH,town_BUKIT PANJANG,...,flat_model_Multi Generation,flat_model_New Generation,flat_model_Premium Apartment,flat_model_Premium Apartment Loft,flat_model_Premium Maisonette,flat_model_Simplified,flat_model_Standard,flat_model_Terrace,flat_model_Type S1,flat_model_Type S2
0,44.0,232000.0,61.333333,11,2017,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,67.0,250000.0,60.583333,2,2017,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
2,67.0,262000.0,62.416667,2,2017,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
3,68.0,265000.0,62.083333,5,2017,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
4,67.0,265000.0,62.416667,2,2017,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False


## 3.2 Train Test Split

We split the prepared data into a training set (used to fit the models) and a test set (kept unseen, used to measure how well the models predict on new data). We hold out 20% for testing and fix `random_state` so the split is reproducible and every model is judged on exactly the same rows.

In [13]:
## Split data into train set and test set

# X = the inputs (everything except price). y = what we want to predict.
col_y = 'resale_price'
X = df.drop([col_y], axis=1)
y = df[col_y]

# Keep 80% for training and set aside 20% for testing.
# random_state makes the split the same every time we run it.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=2026
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

X_train shape: (187484, 55)
X_test shape : (46872, 55)


# 4. Modelling

Here we train several models, compare them fairly, pick the best one, and tune it.

**The models we try.** We start with a simple baseline and then try more powerful models. Each one has to beat the baseline to be worth using:

1. **Linear Regression** (baseline). A simple, fast starting point that fits a straight line.
2. **Decision Tree.** Can capture non linear patterns that a straight line cannot.
3. **Random Forest.** Many decision trees averaged together, usually more accurate and more stable than a single tree.
4. **Gradient Boosting.** Trees built one after another, where each new tree fixes the mistakes of the ones before it.

**How we measure success.** We score every model on the same three metrics so the comparison is fair. Both MAE and RMSE are in dollars, which is exactly what a buyer cares about:

- **MAE (Mean Absolute Error)** is our main metric. It is simply the average dollar amount the estimate is off by, so it is very easy to explain to a buyer: "on average, this estimate is within about $X of the real price."
- **RMSE (Root Mean Squared Error)** is also in dollars, but it punishes big misses more than small ones. We watch it to make sure the model is not occasionally way off, which matters a lot on a purchase this expensive.
- **R2** is a 0 to 1 score showing how much of the price variation the model explains (closer to 1 is better). It gives a quick overall sense of fit.

We train all four models, compare these metrics in a table, pick the best model, and tune it in Section 4.4.

## 4.1 Baseline: Linear Regression

The simplest model we try. It fits a straight line relationship between the features and price. Its scores become the baseline that every other model has to beat.

In [14]:
## Initialise and train model

# Helper function so we train and score every model the same way.
# It also saves each model's scores into 'results' for a comparison table later.
results = []

def evaluate_model(model, name):
    model.fit(X_train, y_train)        # train on the training data
    y_pred = model.predict(X_test)     # predict on the unseen test data

    # Work out the three scores
    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2})
    print(name)
    print(f"  RMSE: ${rmse:,.0f}")
    print(f"  MAE : ${mae:,.0f}")
    print(f"  R2  : {r2:.4f}")
    return model

# Baseline: Linear Regression
linr = LinearRegression()
linr = evaluate_model(linr, "Linear Regression (baseline)")

Linear Regression (baseline)
  RMSE: $69,456
  MAE : $52,871
  R2  : 0.8688


## 4.2 Other models

Now we train the three tree based models. Each one uses `random_state` so the results are the same every run. Random Forest also uses `n_jobs=-1`, which just means "use all CPU cores to train faster".

Note: Random Forest is the slowest to train (about 1 to 3 minutes on a normal laptop) because it builds many trees. This is normal, not a freeze.

In [15]:
# Decision Tree
dt = DecisionTreeRegressor(random_state=2026)
dt = evaluate_model(dt, "Decision Tree")

# Random Forest (n_jobs=-1 uses all CPU cores so it trains faster)
rf = RandomForestRegressor(random_state=2026, n_jobs=-1)
rf = evaluate_model(rf, "Random Forest")

# Gradient Boosting
gbr = GradientBoostingRegressor(random_state=2026)
gbr = evaluate_model(gbr, "Gradient Boosting")

Decision Tree
  RMSE: $52,057
  MAE : $34,336
  R2  : 0.9263
Random Forest
  RMSE: $40,791
  MAE : $27,835
  R2  : 0.9548
Gradient Boosting
  RMSE: $67,857
  MAE : $48,874
  R2  : 0.8748


## 4.3 Compare the models

I put all four models' scores into one table and sort by MAE (lower is better, since MAE is my main metric) to see which one predicts prices most accurately.

In [16]:
# Put all the model scores into one table, sorted by MAE (lower = better)
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
results_df

,Model,RMSE,MAE,R2
0,Random Forest,40791.147410,27834.769996,0.954762
1,Decision Tree,52057.246005,34335.739888,0.926323
2,Gradient Boosting,67856.732049,48874.046853,0.874814
3,Linear Regression (baseline),69455.921043,52870.639526,0.868844


On pure accuracy, Random Forest is the best model, with the lowest error and the highest R2:

| Model | MAE | RMSE | R2 |
| --- | --- | --- | --- |
| Random Forest | ~$27,800 | ~$40,800 | 0.955 |
| Decision Tree | ~$34,300 | ~$52,100 | 0.926 |
| Gradient Boosting (default) | ~$48,900 | ~$67,900 | 0.875 |
| Linear Regression (baseline) | ~$52,900 | ~$69,500 | 0.869 |

But I could not actually deploy Random Forest. My goal is a web app, which means saving the model to a file and putting it on GitHub, and GitHub does not allow files larger than 100 MB. A Random Forest trained on 234k rows saves to a file over 1 GB because it stores many deep trees, and shrinking it by capping the trees makes it much less accurate. So the most accurate model was not usable for me.

I chose Gradient Boosting instead. It builds many shallow trees rather than a few deep ones, so its saved file is only a few MB and deploys easily. Its default settings were weak (MAE ~$48,900), but only because the defaults use too few and too shallow trees, so I expected tuning to bring it close to Random Forest while keeping it small. Since the whole point of my project is a working web app, I picked the model I could actually deploy.

## 4.4 Hyperparameter Tuning (Gradient Boosting)

Gradient Boosting was small enough to deploy but weak with its default settings (MAE ~$48,900), so I tuned it to make it more accurate.

I used RandomizedSearchCV, which lets me control how many setting combinations it tries, and I varied two hyperparameters:

- n_estimators (the number of trees): I tried 300 and 500, because the default of 100 was too few and made the model underfit.
- max_depth (how deep each tree grows): I tried 4, 5 and 6, kept small because boosting works with shallow trees and it also keeps the saved file tiny.

Each combination is scored with 3 fold cross validation on the training data, using MAE. I set random_state so the search gives the same result every time.

In [ ]:
# I tune two hyperparameters, each with a small set of values to try
param_dist = {
    'n_estimators': [300, 500],    # number of trees (default 100 was too few)
    'max_depth': [4, 5, 6],        # how deep each tree grows (kept small = small file)
}

# RandomizedSearchCV scores each combination by MAE with 3 fold cross validation.
# n_jobs=-1 uses all CPU cores. random_state makes the search repeatable.
gbr_search = RandomizedSearchCV(
    estimator=GradientBoostingRegressor(random_state=2026),
    param_distributions=param_dist,
    n_iter=6,
    cv=3,
    scoring='neg_mean_absolute_error',
    random_state=2026,
    n_jobs=-1
)
gbr_search.fit(X_train, y_train)

print("Best settings found:", gbr_search.best_params_)

In [18]:
# Compare every combination the search tried (cross validation MAE on the training folds)
cv_results = pd.DataFrame(gbr_search.cv_results_)
cv_results['cv_MAE'] = -cv_results['mean_test_score']   # scores are stored as negative MAE
cv_results[['param_n_estimators', 'param_max_depth', 'cv_MAE']].sort_values('cv_MAE').reset_index(drop=True)

,param_n_estimators,param_max_depth,cv_MAE
0,500,6,28930.615148
1,300,6,30245.847698
2,500,5,30517.579496
3,300,5,31954.444529
4,500,4,32614.001793
5,300,4,34317.291579


In [19]:
# Score the tuned Gradient Boosting on the unseen test set (this also adds it to the results table)
tuned_gbr = gbr_search.best_estimator_
tuned_gbr = evaluate_model(tuned_gbr, "Gradient Boosting (tuned)")

Gradient Boosting (tuned)
  RMSE: $41,282
  MAE : $29,055
  R2  : 0.9537


The search picked n_estimators=500 and max_depth=6, and tuning made a big difference:

| Model | MAE | RMSE | R2 |
| --- | --- | --- | --- |
| Gradient Boosting (default) | ~$48,900 | ~$67,900 | 0.875 |
| Gradient Boosting (tuned) | ~$29,100 | ~$41,300 | 0.954 |

Tuning cut the MAE from ~$48,900 down to ~$29,100, an improvement of about 40%. The default model was underfit because it used too few and too shallow trees, so adding more trees (500) and a bit more depth (6) let it capture the real patterns in the data.

I kept the tuned Gradient Boosting as my final model. On average its estimates are within about $29,000 of the real price, it explains about 95% of the price variation, and its saved file is only a few MB, so I can actually deploy it. That is almost as accurate as Random Forest (~$27,800) but small enough to use.

# 5. Model Evaluation

We already know our final model (the tuned Gradient Boosting) is accurate and small enough to deploy. Here we look a bit deeper:

1. **Feature importance**: which features the model relies on most.
2. **A test prediction**: we build one brand new flat, like a user would enter in the app, and check that the model gives a sensible price that moves in the right direction when we change the inputs.

In [ ]:
## Feature importance: which features the final model relies on most
importances = pd.Series(tuned_gbr.feature_importances_, index=X_train.columns)
importances.sort_values().tail(15).plot(kind='barh', figsize=(8, 6))
plt.title('Top 15 Most Important Features (Tuned Gradient Boosting)')
plt.xlabel('Importance')
plt.show()
## interpretation:
# Floor area is by far the most important feature (about 39% of the model's decisions),
# followed by transaction year (~20%), storey (~9%) and remaining lease (~8%). The town
# and flat model columns each add a smaller amount on their own, but together they help
# the model price specific locations and premium models (BUKIT MERAH and DBSS flats stand
# out most). This matches what we saw in the EDA and confirms our engineered features
# (storey_mid, remaining_lease_years, txn_year) are genuinely useful.

In [ ]:
## New data: build one example flat, the way a user would enter it in the app
new_flat = pd.DataFrame([{
    'floor_area_sqm': 90,
    'remaining_lease_years': 70,
    'storey_mid': 11,
    'txn_year': 2026,
    'town': 'ANG MO KIO',
    'flat_type': '4 ROOM',
    'flat_model': 'Model A',
}])

## Preprocess it the SAME way as the training data: one hot encode, then line up the columns.
## reindex adds any missing dummy columns as 0, so the shape matches the training features.
new_flat_encoded = pd.get_dummies(new_flat)
new_flat_encoded = new_flat_encoded.reindex(columns=X_train.columns, fill_value=0)

## Predict the price for this flat
predicted_price = tuned_gbr.predict(new_flat_encoded)[0]
print(f"Predicted resale price: ${predicted_price:,.0f}")

## Sanity check: a bigger flat should cost more. We change only the floor area and predict again.
print("\nSanity check (increase floor area, price should go up):")
for area in [60, 90, 120, 150]:
    check = new_flat.copy()
    check['floor_area_sqm'] = area
    check_encoded = pd.get_dummies(check).reindex(columns=X_train.columns, fill_value=0)
    print(f"  {area} sqm -> ${tuned_gbr.predict(check_encoded)[0]:,.0f}")
## interpretation:
# The model gives a sensible price for a normal flat, and it moves the right way when we
# change the inputs: making the flat bigger steadily raises the predicted price. This is an
# important check, because a model that predicted a lower price for a bigger flat would be
# broken and would confuse buyers in the app.

## Iterative model development

Our final model was not built in one shot. We improved it step by step, and every step is shown above:

1. Started with a simple **Linear Regression** baseline (MAE ~$52,900).
2. Trained three stronger models and compared them on the same test set. **Random Forest** was the most accurate (MAE ~$27,800), but its saved file was over 1 GB, far too big to deploy.
3. Chose **Gradient Boosting** instead, because it is tiny (a few MB) and deployable, then **tuned** it with RandomizedSearchCV. Tuning cut its error by about 40% (from ~$48,900 to ~$29,100), making it almost as accurate as Random Forest.

Scoring every version the same way on the same test set keeps the comparison fair. This is the iterative process: start simple, try stronger models, then pick and tune the best one that also fits our real world constraint that it has to deploy.

## Further feature engineering / feature selection

The feature importance chart shows that a few features (floor area, transaction year, storey, remaining lease) do most of the work, while the many town and flat model columns each add only a little on their own.

We considered dropping the low importance columns (feature selection), but chose to keep them, for two reasons:

1. Gradient Boosting is not hurt by extra low signal features, so keeping them does no harm to accuracy.
2. Each town and flat model column still helps the model price that specific category correctly, which matters for a tool that has to work for any flat across all of Singapore.

So no further feature engineering or selection was needed. Our final model stays the tuned Gradient Boosting.

## Save the final model for deployment

We save two things to files: the trained model, and the exact list of feature columns (in order). The Streamlit app loads both to turn a user's inputs into a prediction. The model file is only a few MB, so it fits easily on GitHub (well under the 100 MB limit).

In [22]:
## Save the final model and the column list the app needs
joblib.dump(tuned_gbr, "hdb_price_model.pkl")             # the trained model (a few MB)
joblib.dump(list(X_train.columns), "model_columns.pkl")   # the 55 feature columns, in the right order

print("Saved: hdb_price_model.pkl and model_columns.pkl")

Saved: hdb_price_model.pkl and model_columns.pkl
